In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("CustomerChurnETL") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("Spark started successfully!")
print("Spark version:", spark.version)

Spark started successfully!
Spark version: 4.3.0


In [2]:
# Read cleaned CSV using PySpark
file_path = "data/processed/cleaned_customer_churn.csv"

spark_df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Dataset extracted successfully!")
print("Total rows:", spark_df.count())
print("Total columns:", len(spark_df.columns))

Dataset extracted successfully!
Total rows: 7043
Total columns: 21


In [3]:
# Display the schema
spark_df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [4]:
# Display the first five records
spark_df.show(5, truncate=False)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines   |InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|7590-VHVEG|Female|0            |Yes    |No        |1     |No          |No phone service|DSL            |No            |Yes         |No              |N

In [5]:
# Check missing values in each column
from pyspark.sql.functions import col, sum

missing_values = spark_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in spark_df.columns
])

missing_values.show()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|         0|     0|            0|      0|         0|     0|           0|            0|              0|             0|           0|               0|          0|          0|              0|       0|               0| 

In [6]:
# Remove customerID column
spark_df = spark_df.drop("customerID")

print("Customer ID removed successfully!")
print("Remaining columns:", len(spark_df.columns))

spark_df.show(5)

Customer ID removed successfully!
Remaining columns: 20
+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|    

In [7]:
from pyspark.sql.functions import col, when

spark_df = spark_df.withColumn(
    "Churn",
    when(col("Churn") == "Yes", 1).otherwise(0)
)

spark_df.groupBy("Churn").count().show()

+-----+-----+
|Churn|count|
+-----+-----+
|    1| 1869|
|    0| 5174|
+-----+-----+



In [8]:
spark_df.printSchema()

root
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: integer (nullable = false)



In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Identify categorical columns
categorical_cols = [
    c for c, dtype in spark_df.dtypes
    if dtype == "string"
]

# Convert text categories into numerical indexes
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

# Convert indexes into one-hot encoded vectors
encoder = OneHotEncoder(
    inputCols=[c + "_index" for c in categorical_cols],
    outputCols=[c + "_encoded" for c in categorical_cols]
)

# Create and run the encoding pipeline
encoding_pipeline = Pipeline(stages=indexers + [encoder])

encoding_model = encoding_pipeline.fit(spark_df)
encoded_df = encoding_model.transform(spark_df)

print("Categorical encoding completed!")
print("Categorical columns:", categorical_cols)

Categorical encoding completed!
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [10]:
encoded_df.select(
    "gender",
    "gender_index",
    "gender_encoded",
    "Churn"
).show(5, truncate=False)

+------+------------+--------------+-----+
|gender|gender_index|gender_encoded|Churn|
+------+------------+--------------+-----+
|Female|1.0         |(2,[1],[1.0]) |0    |
|Male  |0.0         |(2,[0],[1.0]) |0    |
|Male  |0.0         |(2,[0],[1.0]) |1    |
|Male  |0.0         |(2,[0],[1.0]) |0    |
|Female|1.0         |(2,[1],[1.0]) |1    |
+------+------------+--------------+-----+
only showing top 5 rows


In [11]:
from pyspark.ml.feature import VectorAssembler

# Numerical columns
numeric_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# Encoded categorical columns
encoded_cols = [
    c + "_encoded" for c in categorical_cols
]

# Combine all features
assembler = VectorAssembler(
    inputCols=numeric_cols + encoded_cols,
    outputCol="features"
)

final_df = assembler.transform(encoded_df)

print("Feature assembly completed!")

final_df.select("features", "Churn").show(5, truncate=False)

Feature assembly completed!
+--------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                              |Churn|
+--------------------------------------------------------------------------------------------------------------------------------------+-----+
|(45,[1,2,3,5,7,8,11,14,16,18,22,24,27,30,33,36,39,41],[1.0,29.85,29.85,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])  |0    |
|(45,[1,2,3,4,6,8,10,12,16,19,21,25,27,30,33,38,40,42],[34.0,56.95,1889.5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0    |
|(45,[1,2,3,4,6,8,10,12,16,19,22,24,27,30,33,36,39,42],[2.0,53.85,108.15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |1    |
|(45,[1,2,3,4,6,8,11,14,16,19,21,25,28,30,33,38,40,43],[45.0,42.3,1840.75,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,

In [12]:
# Split dataset into training and testing sets
train_data, test_data = final_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

print("Data splitting completed!")

Training records: 5698
Testing records: 1345
Data splitting completed!


In [13]:
print("Training dataset:")
train_data.select("features", "Churn").show(3, truncate=False)

print("Testing dataset:")
test_data.select("features", "Churn").show(3, truncate=False)

Training dataset:
+------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                            |Churn|
+------------------------------------------------------------------------------------------------------------------------------------+-----+
|(45,[1,2,3,5,6,8,11,14,16,18,21,24,27,30,33,36,40,43],[1.0,25.25,25.25,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0    |
|(45,[1,2,3,5,6,8,11,14,16,18,21,24,27,30,33,36,40,41],[1.0,24.6,24.6,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])  |1    |
|(45,[1,2,3,5,6,8,11,14,16,18,21,24,27,30,33,36,40,42],[1.0,24.4,24.4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])  |0    |
+------------------------------------------------------------------------------------------------------------------------------------+--

In [14]:
from pyspark.ml.classification import LogisticRegression

# Create Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Churn",
    maxIter=20
)

# Train the model
lr_model = lr.fit(train_data)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [15]:
# Predict customer churn
predictions = lr_model.transform(test_data)

# Display actual and predicted results
predictions.select(
    "Churn",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|Churn|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |1.0       |[0.39453584154290733,0.6054641584570927]|
|1    |1.0       |[0.41487951646942994,0.5851204835305701]|
|0    |0.0       |[0.6319566184194918,0.3680433815805082] |
|0    |0.0       |[0.5915216303752132,0.4084783696247868] |
|1    |0.0       |[0.5383759559551922,0.46162404404480784]|
|0    |0.0       |[0.6306188142178447,0.3693811857821553] |
|0    |0.0       |[0.5616896963669833,0.4383103036330167] |
|1    |0.0       |[0.5592506852795508,0.4407493147204492] |
|0    |0.0       |[0.5717221071876893,0.42827789281231066]|
|1    |0.0       |[0.5716188887213477,0.4283811112786523] |
+-----+----------+----------------------------------------+
only showing top 10 rows


In [16]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Calculate accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

# Calculate precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_evaluator.evaluate(predictions)

# Calculate recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_evaluator.evaluate(predictions)

# Calculate F1 score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_evaluator.evaluate(predictions)

print("Logistic Regression Evaluation")
print("--------------------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

Logistic Regression Evaluation
--------------------------------
Accuracy: 0.8052
Precision: 0.7964
Recall: 0.8052
F1 Score: 0.7985


In [17]:
predictions.groupBy("Churn", "prediction") \
    .count() \
    .orderBy("Churn", "prediction") \
    .show()

+-----+----------+-----+
|Churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|  888|
|    0|       1.0|   98|
|    1|       0.0|  164|
|    1|       1.0|  195|
+-----+----------+-----+



In [18]:
from pyspark.ml.classification import RandomForestClassifier

# Create Random Forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="Churn",
    numTrees=100,
    seed=42
)

# Train the model
rf_model = rf.fit(train_data)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [19]:
# Generate predictions
rf_predictions = rf_model.transform(test_data)

rf_predictions.select(
    "Churn",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|Churn|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |1.0       |[0.4571842020981429,0.5428157979018571] |
|1    |0.0       |[0.5198290202006988,0.48017097979930123]|
|0    |0.0       |[0.6553931136243395,0.34460688637566045]|
|0    |0.0       |[0.6391538452690919,0.3608461547309082] |
|1    |1.0       |[0.47540666044305924,0.5245933395569409]|
|0    |0.0       |[0.5401813059835969,0.45981869401640313]|
|0    |0.0       |[0.5236082441091967,0.4763917558908034] |
|1    |0.0       |[0.5247750980758821,0.4752249019241179] |
|0    |0.0       |[0.6050399902971072,0.3949600097028928] |
|1    |0.0       |[0.6068310038812242,0.39316899611877576]|
+-----+----------+----------------------------------------+
only showing top 10 rows


In [45]:
# Calculate Random Forest confusion matrix values

rf_tp = rf_predictions.filter(
    (col("Churn") == 1) & (col("prediction") == 1)
).count()

rf_fn = rf_predictions.filter(
    (col("Churn") == 1) & (col("prediction") == 0)
).count()

rf_fp = rf_predictions.filter(
    (col("Churn") == 0) & (col("prediction") == 1)
).count()

rf_tn = rf_predictions.filter(
    (col("Churn") == 0) & (col("prediction") == 0)
).count()

# Calculate churn recall
rf_churn_recall = rf_tp / (rf_tp + rf_fn)

print("Random Forest Confusion Matrix:")
print([[rf_tn, rf_fp], [rf_fn, rf_tp]])

print("Random Forest Churn Recall:",
      round(rf_churn_recall * 100, 2), "%")

Random Forest Confusion Matrix:
[[931, 55], [239, 120]]
Random Forest Churn Recall: 33.43 %


In [20]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]

rf_results = {}

for metric in metrics:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="Churn",
        predictionCol="prediction",
        metricName=metric
    )

    rf_results[metric] = evaluator.evaluate(rf_predictions)

print("Random Forest Evaluation")
print("------------------------")
print("Accuracy:", round(rf_results["accuracy"], 4))
print("Precision:", round(rf_results["weightedPrecision"], 4))
print("Recall:", round(rf_results["weightedRecall"], 4))
print("F1 Score:", round(rf_results["f1"], 4))

Random Forest Evaluation
------------------------
Accuracy: 0.7814
Precision: 0.7664
Recall: 0.7814
F1 Score: 0.7531


In [21]:
comparison = [
    ("Logistic Regression", accuracy, precision, recall, f1),
    (
        "Random Forest",
        rf_results["accuracy"],
        rf_results["weightedPrecision"],
        rf_results["weightedRecall"],
        rf_results["f1"]
    )
]

comparison_df = spark.createDataFrame(
    comparison,
    ["Model", "Accuracy", "Precision", "Recall", "F1_Score"]
)

comparison_df.show(truncate=False)

+-------------------+------------------+------------------+------------------+------------------+
|Model              |Accuracy          |Precision         |Recall            |F1_Score          |
+-------------------+------------------+------------------+------------------+------------------+
|Logistic Regression|0.8052044609665427|0.7964415524608166|0.8052044609665427|0.7984993742067636|
|Random Forest      |0.7814126394052044|0.7663626419016754|0.7814126394052044|0.7530808692581382|
+-------------------+------------------+------------------+------------------+------------------+



In [22]:
from pyspark.mllib.evaluation import MulticlassMetrics

# Prepare prediction and actual label pairs
prediction_labels = predictions.select(
    "prediction", "Churn"
).rdd.map(
    lambda row: (float(row["prediction"]), float(row["Churn"]))
)

# Generate confusion matrix
metrics = MulticlassMetrics(prediction_labels)

print("Logistic Regression Confusion Matrix:")
print(metrics.confusionMatrix().toArray())

C:\Users\HP\Documents\Customer-Churn-Prediction\.venv\Lib\site-packages\pyspark\sql\context.py:165: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Logistic Regression Confusion Matrix:
[[888.  98.]
 [164. 195.]]


In [23]:
print("Churn class precision:", metrics.precision(1.0))
print("Churn class recall:", metrics.recall(1.0))
print("Churn class F1-score:", metrics.fMeasure(1.0))

Churn class precision: 0.6655290102389079
Churn class recall: 0.5431754874651811
Churn class F1-score: 0.598159509202454


In [24]:
import os

os.makedirs("outputs/models", exist_ok=True)

lr_model.write().overwrite().save(
    "outputs/models/logistic_regression_model"
)

print("Logistic Regression model saved successfully!")

Py4JJavaError: An error occurred while calling o4005.save.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(

In [25]:
import os
import pickle

os.makedirs("outputs/models", exist_ok=True)

with open("outputs/models/logistic_regression_model.pkl", "wb") as file:
    pickle.dump(lr_model, file)

print("Logistic Regression model saved successfully!")


TypeError: cannot pickle '_thread.RLock' object

In [26]:
import os
import json

os.makedirs("outputs/models", exist_ok=True)

model_parameters = {
    "model_name": "Logistic Regression",
    "coefficients": lr_model.coefficients.toArray().tolist(),
    "intercept": float(lr_model.intercept),
    "number_of_features": lr_model.numFeatures
}

with open(
    "outputs/models/logistic_regression_parameters.json",
    "w"
) as file:
    json.dump(model_parameters, file, indent=4)

print("Model parameters saved successfully!")

Model parameters saved successfully!


In [27]:
os.makedirs("data/processed", exist_ok=True)

final_df.write \
    .mode("overwrite") \
    .parquet("data/processed/churn_features.parquet")

print("Processed PySpark dataset saved successfully!")

Py4JJavaError: An error occurred while calling o4016.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdi

In [28]:
from pyspark.ml.functions import vector_to_array

# Convert Spark feature vectors into normal arrays
final_pandas = final_df.select(
    vector_to_array("features").alias("features"),
    "Churn"
).toPandas()

print("Converted successfully!")
print(final_pandas.shape)

Converted successfully!
(7043, 2)


In [29]:
import os

os.makedirs("data/processed", exist_ok=True)

final_pandas.to_parquet(
    "data/processed/churn_features.parquet",
    index=False
)

print("Parquet file saved successfully!")

PermissionError: [WinError 5] Failed to open local file 'data/processed/churn_features.parquet'. Detail: [Windows error 5] Access is denied.


In [30]:
final_pandas.to_parquet(
    "data/processed/churn_features_v2.parquet",
    index=False
)

print("New Parquet file saved successfully!")

New Parquet file saved successfully!


In [31]:
check_df = pd.read_parquet("data/processed/churn_features_v2.parquet")

print("Rows and columns:", check_df.shape)
print(check_df.head())

NameError: name 'pd' is not defined

In [32]:
import pandas as pd

check_df = pd.read_parquet(
    "data/processed/churn_features_v2.parquet"
)

print("Rows and columns:", check_df.shape)
print(check_df.head())

Rows and columns: (7043, 2)
                                            features  Churn
0  [0.0, 1.0, 29.85, 29.85, 0.0, 1.0, 0.0, 1.0, 1...      0
1  [0.0, 34.0, 56.95, 1889.5, 1.0, 0.0, 1.0, 0.0,...      0
2  [0.0, 2.0, 53.85, 108.15, 1.0, 0.0, 1.0, 0.0, ...      1
3  [0.0, 45.0, 42.3, 1840.75, 1.0, 0.0, 1.0, 0.0,...      0
4  [0.0, 2.0, 70.7, 151.65, 0.0, 1.0, 1.0, 0.0, 1...      1


In [33]:
import pandas as pd
import os

os.makedirs("outputs", exist_ok=True)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [0.8052, 0.7814],
    "Weighted Precision": [0.7964, 0.7664],
    "Weighted Recall": [0.8052, 0.7814],
    "F1 Score": [0.7985, 0.7531]
})

results.to_csv("outputs/model_comparison.csv", index=False)

print(results)
print("\nModel comparison saved successfully!")

                 Model  Accuracy  Weighted Precision  Weighted Recall  \
0  Logistic Regression    0.8052              0.7964           0.8052   
1        Random Forest    0.7814              0.7664           0.7814   

   F1 Score  
0    0.7985  
1    0.7531  

Model comparison saved successfully!


In [34]:
from pyspark.sql.functions import col, when

# Count each class in the training dataset
class_counts = {
    row["Churn"]: row["count"]
    for row in train_data.groupBy("Churn").count().collect()
}

print("Class counts:", class_counts)

# Calculate class weights
total = sum(class_counts.values())

weight_0 = total / (2 * class_counts[0])
weight_1 = total / (2 * class_counts[1])

print("Weight for No Churn:", weight_0)
print("Weight for Churn:", weight_1)

# Add class weights to training data
weighted_train_data = train_data.withColumn(
    "classWeight",
    when(col("Churn") == 1, weight_1).otherwise(weight_0)
)

print("Class balancing completed!")

Class counts: {1: 1510, 0: 4188}


PySparkTypeError: [NOT_EXPECTED_TYPE] Argument `col` should be Column or str, got dict_values.

In [35]:
import builtins
from pyspark.sql.functions import col, when

# Calculate total using Python's built-in sum
total = builtins.sum(class_counts.values())

# Calculate class weights
weight_0 = total / (2 * class_counts[0])
weight_1 = total / (2 * class_counts[1])

print("Total training records:", total)
print("Weight for No Churn:", weight_0)
print("Weight for Churn:", weight_1)

# Apply weights
weighted_train_data = train_data.withColumn(
    "classWeight",
    when(col("Churn") == 1, weight_1).otherwise(weight_0)
)

print("Class balancing completed successfully!")

Total training records: 5698
Weight for No Churn: 0.6802769818529131
Weight for Churn: 1.8867549668874173
Class balancing completed successfully!


In [36]:
from pyspark.ml.classification import LogisticRegression

balanced_lr = LogisticRegression(
    featuresCol="features",
    labelCol="Churn",
    weightCol="classWeight",
    maxIter=30
)

balanced_lr_model = balanced_lr.fit(weighted_train_data)

# Generate predictions on the original test dataset
balanced_predictions = balanced_lr_model.transform(test_data)

print("Balanced Logistic Regression trained successfully!")
balanced_predictions.select(
    "Churn", "prediction", "probability"
).show(10, truncate=False)

Balanced Logistic Regression trained successfully!
+-----+----------+----------------------------------------+
|Churn|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |1.0       |[0.19303538334214732,0.8069646166578527]|
|1    |1.0       |[0.2155971652578347,0.7844028347421653] |
|0    |1.0       |[0.4077340296493927,0.5922659703506072] |
|0    |1.0       |[0.35474435487582007,0.64525564512418]  |
|1    |1.0       |[0.2940818414406017,0.7059181585593983] |
|0    |1.0       |[0.3815475722053745,0.6184524277946255] |
|0    |1.0       |[0.3092885915427491,0.6907114084572509] |
|1    |1.0       |[0.32726265685168354,0.6727373431483165]|
|0    |1.0       |[0.3168828589995574,0.6831171410004426] |
|1    |1.0       |[0.3274651840379307,0.6725348159620693] |
+-----+----------+----------------------------------------+
only showing top 10 rows


In [37]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="Churn",
    predictionCol="prediction"
)

balanced_accuracy = evaluator.evaluate(
    balanced_predictions,
    {evaluator.metricName: "accuracy"}
)

balanced_precision = evaluator.evaluate(
    balanced_predictions,
    {evaluator.metricName: "weightedPrecision"}
)

balanced_recall = evaluator.evaluate(
    balanced_predictions,
    {evaluator.metricName: "weightedRecall"}
)

balanced_f1 = evaluator.evaluate(
    balanced_predictions,
    {evaluator.metricName: "f1"}
)

print("Balanced Logistic Regression Results")
print("Accuracy:", round(balanced_accuracy * 100, 2), "%")
print("Weighted Precision:", round(balanced_precision * 100, 2), "%")
print("Weighted Recall:", round(balanced_recall * 100, 2), "%")
print("F1 Score:", round(balanced_f1 * 100, 2), "%")

Balanced Logistic Regression Results
Accuracy: 74.94 %
Weighted Precision: 80.9 %
Weighted Recall: 74.94 %
F1 Score: 76.27 %


In [38]:
from pyspark.mllib.evaluation import MulticlassMetrics

balanced_labels = balanced_predictions.select(
    "prediction", "Churn"
).rdd.map(
    lambda row: (float(row["prediction"]), float(row["Churn"]))
)

balanced_metrics = MulticlassMetrics(balanced_labels)

print("Confusion Matrix:")
print(balanced_metrics.confusionMatrix().toArray())

print("\nChurn Precision:",
      round(balanced_metrics.precision(1.0) * 100, 2), "%")

print("Churn Recall:",
      round(balanced_metrics.recall(1.0) * 100, 2), "%")

print("Churn F1 Score:",
      round(balanced_metrics.fMeasure(1.0) * 100, 2), "%")

C:\Users\HP\Documents\Customer-Churn-Prediction\.venv\Lib\site-packages\pyspark\sql\context.py:165: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Confusion Matrix:
[[716. 270.]
 [ 67. 292.]]

Churn Precision: 51.96 %
Churn Recall: 81.34 %
Churn F1 Score: 63.41 %


In [39]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Balanced Logistic Regression"
    ],
    "Accuracy": [80.52, 78.14, 74.94],
    "Weighted Precision": [79.64, 76.64, 80.90],
    "Weighted Recall": [80.52, 78.14, 74.94],
    "F1 Score": [79.85, 75.31, 76.27],
    "Churn Recall": [54.32, 54.32, 81.34]
})

results.to_csv("outputs/model_comparison.csv", index=False)

print(results)
print("Updated model comparison saved!")

                          Model  Accuracy  Weighted Precision  \
0           Logistic Regression     80.52               79.64   
1                 Random Forest     78.14               76.64   
2  Balanced Logistic Regression     74.94               80.90   

   Weighted Recall  F1 Score  Churn Recall  
0            80.52     79.85         54.32  
1            78.14     75.31         54.32  
2            74.94     76.27         81.34  
Updated model comparison saved!


In [40]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [
        d for d in dirs
        if d not in [".venv", ".ipynb_checkpoints", "__pycache__"]
    ]

    level = root.count(os.sep)
    print("  " * level + os.path.basename(root) + "/")

    for file in files:
        print("  " * (level + 1) + file)

./
  01_data_exploration.ipynb
  01_setup_test.ipynb
  02_etl_pipeline.ipynb
  data/
    processed/
      churn_features_v2.parquet
      cleaned_customer_churn.csv
      permission_test.txt
      churn_features.parquet/
    raw/
      Telco-Customer-Churn.csv
  outputs/
    model_comparison.csv
    models/
      logistic_regression_model.pkl
      logistic_regression_parameters.json
      logistic_regression_model/


In [41]:
import os

paths = [
    "data/processed/churn_features.parquet",
    "outputs/models/logistic_regression_model.pkl",
    "outputs/models/logistic_regression_model"
]

for path in paths:
    print("\nPath:", path)
    print("Exists:", os.path.exists(path))

    if os.path.isdir(path):
        print("Contents:", os.listdir(path))
    elif os.path.isfile(path):
        print("File size:", os.path.getsize(path), "bytes")


Path: data/processed/churn_features.parquet
Exists: True
Contents: []

Path: outputs/models/logistic_regression_model.pkl
Exists: True
File size: 0 bytes

Path: outputs/models/logistic_regression_model
Exists: True
Contents: []


In [42]:
import os
import shutil

failed_paths = [
    "data/processed/churn_features.parquet",
    "data/processed/permission_test.txt",
    "outputs/models/logistic_regression_model.pkl",
    "outputs/models/logistic_regression_model"
]

for path in failed_paths:
    if os.path.isfile(path):
        os.remove(path)
        print("Deleted file:", path)
    elif os.path.isdir(path):
        shutil.rmtree(path)
        print("Deleted folder:", path)

print("\nCleanup completed!")

Deleted folder: data/processed/churn_features.parquet
Deleted file: data/processed/permission_test.txt
Deleted file: outputs/models/logistic_regression_model.pkl
Deleted folder: outputs/models/logistic_regression_model

Cleanup completed!


In [43]:
print("Processed data:", os.listdir("data/processed"))
print("Model outputs:", os.listdir("outputs/models"))

Processed data: ['churn_features_v2.parquet', 'cleaned_customer_churn.csv']
Model outputs: ['logistic_regression_parameters.json']


In [44]:
readme_content = """
# Customer Churn Prediction

## 1. Project Overview
This project focuses on predicting customer churn using machine learning.
It uses the IBM Telco Customer Churn dataset to identify customers who
are likely to discontinue their telecommunications services.

## 2. Objectives
- Explore and understand customer data.
- Clean and preprocess the dataset.
- Build an ETL pipeline using Python and PySpark.
- Apply machine learning classification algorithms.
- Evaluate and compare model performance.
- Identify customers who are likely to churn.

## 3. Technologies Used
- Python
- Jupyter Notebook
- Pandas
- Matplotlib
- Seaborn
- Apache PySpark
- Machine Learning

## 4. Dataset
IBM Telco Customer Churn dataset.

Original records: 7,043
Original columns: 21

## 5. Project Workflow
1. Data collection
2. Data exploration
3. Data cleaning
4. Data transformation
5. Feature engineering
6. Model training
7. Model evaluation
8. Model comparison

## 6. Machine Learning Models
- Logistic Regression
- Random Forest
- Balanced Logistic Regression

## 7. Model Results

| Model | Accuracy | Churn Recall |
|---|---:|---:|
| Logistic Regression | 80.52% | 54.32% |
| Random Forest | 78.14% | 54.32% |
| Balanced Logistic Regression | 74.94% | 81.34% |

The original Logistic Regression achieved the highest overall
accuracy, while Balanced Logistic Regression achieved the
highest churn recall.

## 8. Project Structure
- data/raw: Original dataset
- data/processed: Cleaned and transformed datasets
- outputs/models: Saved model parameters
- outputs: Model comparison results
- 01_data_exploration.ipynb: Exploratory data analysis
- 02_etl_pipeline.ipynb: ETL and machine learning pipeline

## 9. Conclusion
The project demonstrates how PySpark and machine learning
can be used to analyse customer data and predict churn.
The results show the trade-off between overall accuracy
and identifying customers likely to leave.
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme_content)

print("README.md created successfully!")

README.md created successfully!


In [46]:
from pyspark.sql.functions import col

# Confusion matrix
rf_predictions.groupBy("Churn", "prediction") \
    .count() \
    .orderBy("Churn", "prediction") \
    .show()

# Churn recall
rf_tp = rf_predictions.filter(
    (col("Churn") == 1) & (col("prediction") == 1)
).count()

rf_fn = rf_predictions.filter(
    (col("Churn") == 1) & (col("prediction") == 0)
).count()

print("Random Forest Churn Recall:",
      round(rf_tp / (rf_tp + rf_fn) * 100, 2), "%")

+-----+----------+-----+
|Churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|  931|
|    0|       1.0|   55|
|    1|       0.0|  239|
|    1|       1.0|  120|
+-----+----------+-----+

Random Forest Churn Recall: 33.43 %


In [47]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Balanced Logistic Regression"
    ],
    "Accuracy": [80.52, 78.14, 74.94],
    "Weighted Precision": [79.64, 76.64, 80.90],
    "Weighted Recall": [80.52, 78.14, 74.94],
    "F1 Score": [79.85, 75.31, 76.27],
    "Churn Recall": [54.32, 33.43, 81.34]
})

results.to_csv("outputs/model_comparison.csv", index=False)

print("Updated model comparison successfully!")
display(results)

Updated model comparison successfully!


,Model,Accuracy,Weighted Precision,Weighted Recall,F1 Score,Churn Recall
0,Logistic Regression,80.52,79.64,80.52,79.85,54.32
1,Random Forest,78.14,76.64,78.14,75.31,33.43
2,Balanced Logistic Regression,74.94,80.90,74.94,76.27,81.34
